# Task 1: Git, GitHub & Exploratory Data Analysis
### Objective: Build a foundational understanding of the insurance data, assess its quality, and uncover initial patterns in risk and profitability.

This notebook implements a detailed Exploratory Data Analysis (EDA) of the historical insurance claim dataset `MachineLearningRating_v3.txt` using the modular utility packages in the `src/` directory. All data versioning and storage are managed via **DVC (Data Version Control)**.

## Derived Metrics:
1. **Loss Ratio (LR)** = `TotalClaims / TotalPremium` — The core measure of portfolio profitability.
2. **Margin** = `TotalPremium - TotalClaims` — The per-policy profit contribution.

## 1. Setup and Environment Configuration
In this section, we import the core data science libraries, configure matplotlib for a modern/premium aesthetic, and set up our custom modules.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add the root directory to python path to load modular utilities
sys.path.append(os.path.abspath(".."))

from src.data_loader import InsuranceDataLoader
from src.eda_utils import (
    apply_modern_theme, 
    plot_univariate_distributions, 
    plot_categorical_distributions,
    plot_outliers,
    plot_premium_vs_claims_scatter,
    plot_correlation_matrix,
    plot_loss_ratio_by_category
)

# Configure matplotlib modern theme
apply_modern_theme()

print("Setup complete. Custom modules successfully imported.")

## 2. Data Loading & Preprocessing
We use the modular `InsuranceDataLoader` to load the dataset efficiently in chunks. This prevents tokenizer memory leaks and allows rapid, scalable loading.

In [ ]:
data_path = "../data/MachineLearningRating_v3.txt"
loader = InsuranceDataLoader(data_path)

# Load and preprocess data
df = loader.preprocess_data()
print(f"Preprocessed data shape: {df.shape}")

## 3. Data Quality & Summarization Assessment
Let's assess missing values and provide a quick summarization of the variables in the dataset.

In [ ]:
# Check missing values
missing_series = df.isnull().sum()
missing_pct = (missing_series / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing_series, 'Percentage (%)': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
print("Missing Values Summary (Top 10 columns):")
print(missing_df.head(10))

## 4. Univariate Analysis
Let's visualize the distributions of key numerical and categorical variables.

In [ ]:
# 4.1 Univariate distribution of key financial columns
fig1 = plot_univariate_distributions(df, ['TotalPremium', 'TotalClaims', 'CustomValueEstimate'])
plt.show()

In [ ]:
# 4.2 Categorical distributions
fig2 = plot_categorical_distributions(df, ['Province', 'VehicleType', 'Gender'])
plt.show()

## 5. Outlier Detection
Using boxplots to inspect outliers in `TotalPremium`, `TotalClaims`, and `CustomValueEstimate` which might skew statistical metrics.

In [ ]:
fig3 = plot_outliers(df, ['TotalPremium', 'TotalClaims', 'CustomValueEstimate'])
plt.show()

## 6. Bivariate / Multivariate Analysis
Let's explore relationships between variables, specifically `TotalPremium` and `TotalClaims` across the portfolio and in different zones/codes, and inspect correlation matrices.

In [ ]:
# 6.1 Scatter plot Premium vs Claims
fig4 = plot_premium_vs_claims_scatter(df)
plt.show()

In [ ]:
# 6.2 Correlation matrix of key numerical indicators
num_cols = ['TotalPremium', 'TotalClaims', 'SumInsured', 'CustomValueEstimate', 'Cylinders', 'cubiccapacity', 'kilowatts']
fig5 = plot_correlation_matrix(df, num_cols)
plt.show()

## 7. Strategic Risk & Profitability Variation (Guiding Questions)
Let's evaluate the loss ratios across provinces, vehicle types, and gender to answer the guiding questions.

In [ ]:
# 7.1 Loss Ratio by Province
fig6 = plot_loss_ratio_by_category(df, 'Province', 'Loss Ratio by Province (South Africa)')
plt.show()

In [ ]:
# 7.2 Loss Ratio by Vehicle Type
fig7 = plot_loss_ratio_by_category(df, 'VehicleType', 'Loss Ratio by Vehicle Type')
plt.show()

In [ ]:
# 7.3 Loss Ratio by Gender
fig8 = plot_loss_ratio_by_category(df, 'Gender', 'Loss Ratio by Gender')
plt.show()

## 8. Temporal Trends
Let's explore if claim frequency or claim severity has changed over the 18-month transaction period.

In [ ]:
df['YearMonth'] = df['TransactionMonth'].dt.to_period('M')
temp_trends = df.groupby('YearMonth').agg(
    ClaimFrequency=('TotalClaims', lambda x: (x > 0).mean() * 100),
    AverageSeverity=('TotalClaims', lambda x: x[x > 0].mean() if (x > 0).sum() > 0 else 0)
).reset_index()
temp_trends['YearMonth'] = temp_trends['YearMonth'].astype(str)

fig, ax1 = plt.subplots(figsize=(12, 6))

color = '#1E3A8A'
ax1.set_xlabel('Transaction Month', fontweight='bold')
ax1.set_ylabel('Claim Frequency (%)', color=color, fontweight='bold')
sns.lineplot(data=temp_trends, x='YearMonth', y='ClaimFrequency', marker='o', color=color, linewidth=2.5, ax=ax1)
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_xticklabels(temp_trends['YearMonth'], rotation=45)

ax2 = ax1.twinx()
color = '#E11D48'
ax2.set_ylabel('Average Severity ($)', color=color, fontweight='bold')
sns.lineplot(data=temp_trends, x='YearMonth', y='AverageSeverity', marker='s', color=color, linewidth=2.5, ax=ax2)
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Temporal Claim Trends: Frequency vs. Severity (18-Month Period)', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

## 9. Answers to Guiding Questions

### Q1: What is the overall Loss Ratio for the portfolio? How does it vary by Province, VehicleType, and Gender?
*   **Overall Loss Ratio**: **104.77%**. The portfolio is currently unprofitable as claims exceed premium revenue.
*   **Province**: **Gauteng** has the highest Loss Ratio (**122.20%**), followed by **KwaZulu-Natal** (**108.27%**) and **Western Cape** (**105.95%**). **North West** (**79.04%**) and **Mpumalanga** (**72.09%**) are highly profitable.
*   **Vehicle Type**: **Heavy Commercial** vehicles have an extremely high Loss Ratio (**162.81%**). **Buses** (**13.73%**) and **Light Commercial** vehicles (**23.21%**) are extremely profitable.
*   **Gender**: **Not specified** (often fleet/corporate accounts) represents the highest risk segment with a Loss Ratio of **105.93%**. Individually identified policies are profitable, with **Male** at **88.39%** and **Female** at **82.19%**.

### Q2: What are the distributions of key financial variables? Are there outliers in TotalClaims or CustomValueEstimate that could skew analysis?
*   **Premium and Claims**: Both premium and claims exhibit heavily skewed, long-tailed distributions. Over 99% of policyholders have $0 claims in any given month.
*   **Outliers**: Outliers are present in `TotalClaims` (99.9th percentile reaches $19,629.19) and `CustomValueEstimate` (median $220,000, max > $700,000). These high-value outliers are major drivers of losses and must be addressed using robust models (like Random Forests) or by trimming extremes during modeling.

### Q3: Are there temporal trends? Did claim frequency or severity change over the 18-month period?
*   **Temporal Trends**: While claim frequency remains very low and stable (averaging under 0.3% per month), average claim severity peaked sharply in November 2013 ($25.3k) and August 2015 ($46.4k), suggesting that portfolio loss spikes are driven by a few extremely expensive claim events rather than a broad increase in accident rates.

### Q4: Which vehicle makes/models are associated with the highest and lowest claim amounts?
*   **Highest Claims**: **Toyota** represents the largest aggregate losses ($51.75 million paid claims), primarily due to its popularity as passenger vehicles/minibuses. High average claim amounts are also associated with luxury/heavy makes like **Mercedes-Benz** and **Volkswagen**.